In [15]:
import json
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
import plotly.graph_objs as go
import plotly.offline as pyo
import ipywidgets as widgets
from IPython.display import display, clear_output

pyo.init_notebook_mode(connected=True)

In [19]:
# Load data
with open("./output/parsed_data_raw.json", "r") as file:
    data = json.load(file)

# Create dropdown for country selection
country_dropdown = widgets.Dropdown(
    options=sorted(data.keys()),
    description='Country:',
    layout=widgets.Layout(width='50%')
)

# Output widgets: one for chart, one for min/max text
output_chart = widgets.Output()
output_text = widgets.Output()

def plot_country(country):
    x = []
    total = []
    long = []
    permanent = []
    asyl = []

    for date in sorted(data[country].keys(), key=lambda d: pd.to_datetime(d, format="%m.%Y", errors="coerce")):
        x.append(date)
        total.append(data[country][date].get('total', {}).get('celkem', 0))
        long.append(data[country][date].get('přechodně', {}).get('celkem', 0))
        permanent.append(data[country][date].get('trvale', {}).get('celkem', 0))
        asyl.append(data[country][date].get('dočasná ochrana', {}).get('celkem', 0))   

    x_dates = pd.to_datetime(x, format="%m.%Y", errors="coerce")
    df = pd.DataFrame({
        "Date": x_dates, 
        "total": total, 
        "long": long, 
        "permanent": permanent, 
        "asyl": asyl
    }).sort_values("Date")

    # Plotly traces (no min/max markers now)
    trace_total = go.Scatter(x=df["Date"], y=df["total"], mode='lines+markers', name='Total', text=df["total"], hoverinfo='x+y+text')
    trace_long = go.Scatter(x=df["Date"], y=df["long"], mode='lines+markers', name='Long', text=df["long"], hoverinfo='x+y+text')
    trace_permanent = go.Scatter(x=df["Date"], y=df["permanent"], mode='lines+markers', name='Permanent', text=df["permanent"], hoverinfo='x+y+text')
    trace_asyl = go.Scatter(x=df["Date"], y=df["asyl"], mode='lines+markers', name='Asyl', text=df["asyl"], hoverinfo='x+y+text')

    layout = go.Layout(
        title=f"Migration Data for {country}",
        xaxis=dict(title='Date'),
        yaxis=dict(title='Value'),
        width=1400,
        height=600
    )

    fig = go.Figure(data=[trace_total, trace_long, trace_permanent, trace_asyl], layout=layout)

    with output_chart:
        clear_output(wait=True)
        pyo.iplot(fig)

    # Calculate min/max per year for "total"
    df['Year'] = df['Date'].dt.year
    summary_rows = []
    for year in sorted(df['Year'].dropna().unique()):
        df_year = df[df['Year'] == year]
        min_val = df_year['total'].min()
        max_val = df_year['total'].max()
        summary_rows.append(f"{year}: Min total = {min_val}, Max total = {max_val}")

    summary_text = "\n".join(summary_rows)

    with output_text:
        clear_output(wait=True)
        print("Min and Max Total Values per Year:\n")
        print(summary_text)

def on_country_change(change):
    plot_country(change['new'])

country_dropdown.observe(on_country_change, names='value')

display(country_dropdown)
display(output_chart)
display(output_text)

# Initial plot
plot_country(country_dropdown.value)

Dropdown(description='Country:', layout=Layout(width='50%'), options=('Afghánistán', 'Albánie', 'Alžírsko', 'A…

Output()

Output()